# Counter-intuitive flows in 2026

This analysis follows Iacopo Bertelli's 3 [part series](https://www.linkedin.com/pulse/counterintuitive-flows-2023-part-3-iacopo-bertelli-xjetf) to examine counter-intuitive (CI) flows across European market regions in 2026.

For an introduction to what CI flows are, why they occur, and how they can be analysed, we highly recommend reading Bertelli's posts first, as this notebook assumes that the reader is familiar with...

Furthermore, notebook builds on top of the VoltPath API we created to make analyses like this reproducible. 

Hence, **before you run this:** sign up at [voltpath.ai](https://voltpath.ai) for a free personal API key (find it at [voltpath.ai/settings](https://voltpath.ai/settings) once logged in), then copy `.env.example` to `.env` and paste it in as `VOLTPATH_API_KEY`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import plotly.graph_objects as go
from common import (
    FLOW_COLORS,
    border_ccr_mechanism,
    fetch_flows_range,
    flow_type_grouped_bar,
    flow_type_shares,
    flow_type_stacked_bar,
    hourly_border_counts,
    is_core_ahc_vbz,
    peak,
    save_chart,
)
from plotly.subplots import make_subplots

## MTU with the most CI flows in 2026

To get a understanding of what we're looking at, let's start with identifying 2026's Market Time Unit (MTU) with the most CI flows. We can visually confirm the output of the cell below using the VoltPath dashboard 

Set the date range to [2026 May 17 LINK HALO]() and drag the playback control to 6:00 UTC, since it defaults to midnight, and observe the swarm of grey arrows we use to denote CI flows:

<img src="media/ci-peak-2026-05-17-dashboard.png" alt="VoltPath dashboard showing peak counter-intuitive flow on 2026-05-17" width="800">

The data was obtained via our `GET /api/v1/flows` endpoint which already joins physical flow with day-ahead prices on both sides of the border to classify each observation into `Convergence`, `Congestion`, `Counter-intuitive`, or `Outage` &ndash; see [Flow Types](https://docs.voltpath.ai/concepts/flow-types) for the full definitions.


In [ ]:
flows_2026 = fetch_flows_range("2026-01-01", "2027-01-01")
peak_row = peak(hourly_border_counts(flows_2026, "Counter-intuitive"))
print(f"Peak: {peak_row['n_borders']} borders counter-intuitive at {peak_row['datetime']}")

## CI flows across Capacity Calculation Regions

Capacity Calculation Regions (CCRs) determine geographical areas within
which TSOs first develop a methodology, implement it, and then compute crosszonal capacities in a coordinated fashion across a set of bidding zone borders following the previously developed methodology.

In the so-called Core CCR &ndash; renamed as Central Europe (CE) in 2024 with the addition of Italy North (and the planned SEM-FR??) &ndash; electricity prices are determined via flow-based market coupling while other regions adopt...

We'll be particularly interested in regions like Hansa and Nordic where significant changes took place in coupling methodology over the past X years.

### Switzerland

Explicit allocation, meshed grid — not an official ACER CCR, coordinated bilaterally.

TODO - need a comment here stating that nothing changed, talks between CH and EU are taking forever

In [ ]:
subset = flows_2026[flows_2026["border"].map(border_ccr_mechanism) == "Switzerland (explicit allocation)"]
fig = flow_type_stacked_bar(flow_type_shares(subset), title="Flow types per border, 2026 — Switzerland")
fig.show()
save_chart(fig, "flow-types-2026-switzerland")

### SWE, GRIT, SEE, Baltic

Price-coupled NTC, where allocation is radial. There was no shared mechanism change in 2026, in fact, the distribution of flows matches observations from previous years.

In [ ]:
radial_ccrs = ["SWE", "GRIT", "SEE", "Baltic"]
groups = [
    (
        ccr,
        flow_type_shares(
            flows_2026[flows_2026["border"].map(border_ccr_mechanism) == f"{ccr} (price-coupled NTC, radial)"]
        ),
    )
    for ccr in radial_ccrs
]
fig = flow_type_grouped_bar(groups, title="Flow types per border, 2026 — NTC radial (SWE, GRIT, SEE, Baltic)")
fig.show()
save_chart(fig, "flow-types-2026-ntc-radial")

GR->IT_SUD is the 1 clear outlier out of 13 borders with over a fifth of its flows showing up as an outage.

We can some insight as to whether these were planned or unplanned though IPTO's [Interconnections Maintenance Schedule](https://www.admie.gr/sites/default/files/attached-files/type-file/2025/12/20260101_20261231_InterconnectionsMaintenanceSchedule_01.pdf). The GRITA line (Arachthos-Galatina) is marked with a ~47-day continuous ("Permanent") planned outage.

### Core

Flow-based market coupling (FBMC), Central Europe — includes Italy North's borders (merged since 19 Mar 2024).

In [ ]:
subset = flows_2026[flows_2026["border"].map(border_ccr_mechanism) == "Core (price-coupled, flow-based)"]
fig = flow_type_stacked_bar(flow_type_shares(subset), title="Flow types per border, 2026 — Core")
fig.show()
save_chart(fig, "flow-types-2026-core")

In [ ]:
core = flows_2026[flows_2026["border"].map(border_ccr_mechanism) == "Core (price-coupled, flow-based)"]
core_before = core[core["datetime"] < "2026-06-11"]
core_after = core[core["datetime"] >= "2026-06-11"]

title_before = "Flow types per border, 2026 — Core (before AHC, until 11 Jun)"
fig = flow_type_stacked_bar(flow_type_shares(core_before), title=title_before)
fig.show()
save_chart(fig, "flow-types-2026-core-before-ahc")

title_after = "Flow types per border, 2026 — Core (after AHC, from 11 Jun)"
fig = flow_type_stacked_bar(flow_type_shares(core_after), title=title_after)
fig.show()
save_chart(fig, "flow-types-2026-core-after-ahc")

In [ ]:
vbz = flows_2026[flows_2026["border"].map(is_core_ahc_vbz)]
vbz_before = vbz[vbz["datetime"] < "2026-06-11"]
vbz_after = vbz[vbz["datetime"] >= "2026-06-11"]

title_before = "Flow types per border, 2026 — Core AHC virtual bidding zones (before AHC, until 11 Jun)"
fig = flow_type_stacked_bar(flow_type_shares(vbz_before), title=title_before)
fig.show()
save_chart(fig, "flow-types-2026-core-vbz-before-ahc")

title_after = "Flow types per border, 2026 — Core AHC virtual bidding zones (after AHC, from 11 Jun)"
fig = flow_type_stacked_bar(flow_type_shares(vbz_after), title=title_after)
fig.show()
save_chart(fig, "flow-types-2026-core-vbz-after-ahc")

Advanced Hybrid Coupling (AHC) replaces Standard Hybrid Coupling (SHC) on Core's external borders only &ndash; under SHC, the exchange forecast on those borders was a static D-2 input to the flow-based clearing; under AHC the interconnector end is modelled as a virtual bidding zone whose net position is solved jointly with the rest of Core, competing for CNEC capacity instead of holding a fixed priority slot ([JAO go-live notice](https://www.jao.eu/news/go-live-date-core-advanced-hybrid-coupling)). Internal Core borders were never SHC-governed, so the mechanism itself doesn't change for them &ndash; see the before/after split below for what did move regardless.

BE->DE_LU's 4.3% zero-nomination share traces to a single dated, non-AHC event: a forced 24h outage on ALEGrO &ndash; the sole tie-line between Belgium and Germany/Luxembourg &ndash; on 28 Jun 2026 ([JAO curtailment notice](https://www.jao.eu/news/capacity-curtailment-be-de-border-delivery-period-2)). ALEGrO's own 1-year uptime record (98%, [Amprion](https://www.amprion.net/Press/Press-Detail-Page_37505.html)) makes this a normal, low-frequency event for a single-cable HVDC link, not a mechanism-change artifact.

**Before/after split, read with caution.** Convergence rises across almost every internal Core border after 11 Jun, a fairly uniform +9 to +14pp jump (e.g. AT->DE_LU 7.7%->20.7%, BE->FR 8.7%->22.5%) &ndash; while the three IT_NORD borders, the only ones genuinely external to Core, barely move (IT_NORD->SI even ticks down). That uniformity across otherwise-unrelated borders looks more like a seasonal effect riding along with the mechanism switch (before = Jan&ndash;10 Jun, after = 11 Jun&ndash;year end) than an AHC effect, and it cuts the opposite way from our own season-controlled &plusmn;4-day comparison around go-live, which found convergence *falling* (21.5%->18.1%), not rising. A single full-year before/after split isn't a clean read on AHC without controlling for season.

One real outlier: BE->NL congestion share nearly doubles, 25.4%->47.3%, far beyond any other border's move. No primary source found discusses this border specifically &ndash; worth a dedicated look if it matters.

Also worth flagging: AHC is reported to have grown Core's bidding-zone model from 14 to 23 by adding virtual bidding zones for the Hansa/Nordic HVDC links plus DE-DK1, PL-LT and RO-BG. Our own border-to-mechanism mapping doesn't reflect this yet &ndash; those borders are still tagged under their pre-AHC regional mechanism (Hansa, Baltic radial, SEE radial respectively), not as Core. The split above only covers the original 22-border Core set; it says nothing about the new virtual bidding zones themselves.

In [ ]:
core = flows_2026[flows_2026["border"].map(border_ccr_mechanism) == "Core (price-coupled, flow-based)"]
shares = flow_type_shares(core)

total_hours = core["duration_h"].sum()
convergence_pct = core[core["flow_type"] == "Convergence"]["duration_h"].sum() / total_hours * 100
congestion_pct = core[core["flow_type"] == "Congestion"]["duration_h"].sum() / total_hours * 100
ci_min, ci_max = shares["Counter-intuitive"].min(), shares["Counter-intuitive"].max()

print(f"Price convergence occurs {convergence_pct:.0f}% of the time across Core borders")
print(f"Congestion occurs {congestion_pct:.0f}% of the time across Core borders")

top_congested = shares.sort_values("Congestion", ascending=False).head(2)
print(
    f"Most congested border: {top_congested.iloc[0]['border']} ({top_congested.iloc[0]['Congestion']:.1f}%), "
    f"followed by {top_congested.iloc[1]['border']} ({top_congested.iloc[1]['Congestion']:.1f}%)"
)

print(f"Counter-intuitive flows occur {ci_min:.0f}-{ci_max:.0f}% of the time, depending on border")

ci_cols = ["border", "Counter-intuitive"]
least_ci = shares.sort_values("Counter-intuitive").head(5)[ci_cols].reset_index(drop=True)
most_ci = shares.sort_values("Counter-intuitive", ascending=False).head(5)[ci_cols].reset_index(drop=True)

fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "table"}, {"type": "table"}]],
    subplot_titles=["5 least counter-intuitive Core borders", "5 most counter-intuitive Core borders"],
)
for col, df in enumerate([least_ci, most_ci], start=1):
    fig.add_trace(
        go.Table(
            header={
                "values": ["Border", "Counter-intuitive %"],
                "fill_color": FLOW_COLORS["Counter-intuitive"],
            },
            cells={"values": [df["border"], df["Counter-intuitive"].round(1)]},
        ),
        row=1,
        col=col,
    )
fig.update_layout(title="Counter-intuitive flows, 2026 — Core borders, least vs. most")
fig.show()
save_chart(fig, "ci-leaderboard-2026-core")

### Nordic

Flow-based market coupling (FBMC) since 29 Oct 2024 — NTC with lineset constraints before that.

In [ ]:
subset = flows_2026[flows_2026["border"].map(border_ccr_mechanism) == "Nordic (price-coupled NTC, lineset)"]
fig = flow_type_stacked_bar(flow_type_shares(subset), title="Flow types per border, 2026 — Nordic")
fig.show()
save_chart(fig, "flow-types-2026-nordic")

### Hansa

NTC with lineset constraints until 11 Jun 2026, then Advanced Hybrid Coupling (AHC) &ndash; split below into before/after to isolate the mechanism change.

In [ ]:
hansa = flows_2026[flows_2026["border"].map(border_ccr_mechanism) == "Hansa (price-coupled NTC, lineset)"]
hansa_before = hansa[hansa["datetime"] < "2026-06-11"]
hansa_after = hansa[hansa["datetime"] >= "2026-06-11"]

title_before = "Flow types per border, 2026 — Hansa (before AHC, until 11 Jun)"
fig = flow_type_stacked_bar(flow_type_shares(hansa_before), title=title_before)
fig.show()
save_chart(fig, "flow-types-2026-hansa-before-ahc")

title_after = "Flow types per border, 2026 — Hansa (after AHC, from 11 Jun)"
fig = flow_type_stacked_bar(flow_type_shares(hansa_after), title=title_after)
fig.show()
save_chart(fig, "flow-types-2026-hansa-after-ahc")

**Data source:** [ENTSO-E Transparency Platform](https://transparency.entsoe.eu), via the [Voltpath API](https://voltpath.ai) (docs at [docs.voltpath.ai](https://docs.voltpath.ai) &ndash; if you're using an AI coding agent with the Context7 MCP server, it can pull these docs directly). Physical flow data is CC-BY 4.0; day-ahead prices are reused under ENTSO-E's general Transparency Platform terms.